In [ ]:
import pandas as pd
import numpy as np


In [ ]:
data=pd.read_csv('/content/mission 4.csv')

In [ ]:
data.head(10)

In [ ]:
data.info()

In [ ]:
for i in data.columns:
  print(data[i].unique())
  print(i)
  print('============')

In [ ]:
data.columns

In [ ]:
from sklearn.preprocessing import LabelEncoder , StandardScaler

In [ ]:
le={}
encode=['Brand','Processor Type','Operating System']
for col in encode:
  le[col]=LabelEncoder()
  data[col]=le[col].fit_transform(data[col])
data.head(10)

In [ ]:
Sc=StandardScaler()
Scaller=['RAM Size (GB)',	'Storage (GB)',	'Screen Size (inches)']
data[Scaller]=Sc.fit_transform(data[Scaller])
data.head(10)

In [ ]:
#Hist Gram for Price , ScatterPlot for Ram and Price , BarPlot for Brand and Price

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(10,8))
sns.histplot(data['Price ($)'],kde=True,color='Red')
plt.title='Price'
plt.xlabel='Price'
plt.ylabel='Freqauncy'
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
sns.scatterplot(x=data['RAM Size (GB)'],y=data['Price ($)'],color="blue")
plt.title='Price'
plt.xlabel='Ram'
plt.ylabel='Price'
plt.show()

In [ ]:
plt.figure(figsize=(10,8))

sns.barplot(x=data['Brand'],y=data['Price ($)'],color="blue")
plt.xlabel='Brand'
plt.ylabel='Price'
plt.show()

In [ ]:
x=data.drop('Price ($)',axis=1)
y=data['Price ($)']

In [ ]:
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error as MAE,r2_score as R2 , mean_squared_error as MSE

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42)

In [ ]:
Lin=LinearRegression()
DTR=DecisionTreeRegressor()
RFR=RandomForestRegressor()

In [ ]:
Lin.fit(x_train,y_train)
DTR.fit(x_train,y_train)
RFR.fit(x_train,y_train)

In [ ]:
y_pred_Lin=Lin.predict(x_test)
y_pred_DTR=DTR.predict(x_test)
y_pred_RFR=RFR.predict(x_test)

In [ ]:
MSE_Lin=MSE(y_test,y_pred_Lin)
MSE_DTR=MSE(y_test,y_pred_DTR)
MSE_RFR=MSE(y_test,y_pred_RFR)

In [ ]:
print(MSE_DTR)
print(MSE_Lin)
print(MSE_RFR)

In [ ]:
MAE_Lin=MAE(y_test,y_pred_Lin)
MAE_DTR=MAE(y_test,y_pred_DTR)
MAE_RFR=MAE(y_test,y_pred_RFR)
print(MAE_DTR)
print(MAE_Lin)
print(MAE_RFR)

In [ ]:
R2_Lin=R2(y_test,y_pred_Lin)
R2_DTR=R2(y_test,y_pred_DTR)
R2_RFR=R2(y_test,y_pred_RFR)
print(R2_DTR)
print(R2_Lin)
print(R2_RFR)

In [ ]:
Grid_Prams={'max_depth':[5,10,15,20,25,30],'max_leaf_nodes':[5,7,10]}
GridModel=GridSearchCV(DecisionTreeRegressor(),Grid_Prams,cv=5)
GridModel.fit(x_train,y_train)
print(GridModel.best_params_)

In [ ]:
y_pred_Grid=GridModel.best_estimator_.predict(x_test)
MSE=MSE(y_test,y_pred_Grid)
R2=R2(y_test,y_pred_Grid)
MAE=MAE(y_test,y_pred_Grid)
print(MSE)
print(R2)
print(MAE)

In [ ]:
Rand_Prams={'n_estimators':[5,10,20,15],'max_depth':[5,10,20,15,25]}
RandModel=RandomizedSearchCV(RandomForestRegressor(),Rand_Prams,cv=5)
RandModel.fit(x_train,y_train)
print(RandModel.best_params_)
print(RandModel.best_score_)

In [ ]:
pip install gradio

In [ ]:
data.columns

In [ ]:
for i in data.columns:
  print(data[i].unique())
  print(i)
  print("------------------")

In [ ]:
import gradio as gr
def Predict(Brand, Processor_Type, RAM_Size, Storage, Screen_Size, Operating_System):
  try:
    inputdata=pd.DataFrame({
        'Brand':[Brand],
        'Processor Type':[Processor_Type],
        'RAM Size (GB)':[RAM_Size],
        'Storage (GB)':[Storage],
        'Screen Size (inches)':[Screen_Size],
        'Operating System':[Operating_System]
    })
    for col in encode:
      inputdata[col]=le[col].transform(inputdata[col])
    inputdata[Scaller]=Sc.transform(inputdata[Scaller])
    result= RFR.predict(inputdata)
    return f'result: ${result[0]:.2f}'
  except Exception as e:
    return str(e)
gr.Interface(
    fn=Predict,
    title='Laptop Price',
    inputs=[
        gr.Dropdown(choices=['Acer','Apple','HP','Lenovo','Dell'],label='Brand'),
        gr.Dropdown(choices=['Intel Core i5','Intel Core i7','AMD Ryzen 5','AMD Ryzen 7'],label='Processor Type'),
        gr.Number(label='Ram Size'),
        gr.Number(label='Storage Size'),
        gr.Number(label='Screen Size'),
        gr.Dropdown(choices=['macOS','Linux','Windows'],label='Operating System')
    ],
    outputs=gr.Textbox(label='Laptop Price')
).launch(debug=True)